# 02 · Pronóstico y backtest rolling-origin (PLAN §3)

Cuatro modelos (Logit-ETS, Logit-ETS damped, factor de reparto, naive estacional), validados con *expanding window*: cada mes objetivo se entrena **solo** con datos anteriores. Requiere `data/raw/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.etl.load import load_migration, load_occupancy, load_survey
from src.etl.clean import clean_migration, clean_occupancy, clean_survey
from src.config import SURVEY_BORDER_MAP, IZABAL_BORDERS

pd.set_option("display.width", 140)
from src.config import ISABAL_FEEDER_BORDERS, VISITOR_TYPES
from src.etl import transform as t
from src.models import forecast as fc

In [ ]:
occ_df = t.occupancy_series(clean_occupancy(load_occupancy()))
sur = clean_survey(load_survey())
mig = clean_migration(load_migration())

occ = fc.como_serie(occ_df, "indice_ocupacion")
enc = fc.serie_mensual_encuesta(t.survey_izabal_daily(sur))
feeders = ISABAL_FEEDER_BORDERS or sorted(SURVEY_BORDER_MAP.values())
migra = fc.serie_mensual_migracion(t.migration_monthly(mig), feeders, VISITOR_TYPES)
print("ocupación:", occ.index.min().date(), "→", occ.index.max().date(), "· encuesta:", enc.index.min().date(), "→", enc.index.max().date(), "· NaN:", int(enc.isna().sum()))

## A. Ocupación de Izabal (acotada a [0, 1] → logit)
Historia mínima de ETS: 25 meses ⇒ con 36 meses quedan 11 folds (feb–dic 2025).

In [ ]:
bt_occ = fc.backtest(occ, transform="logit", bounds=(0, 1), migracion=migra)
met_occ = fc.metricas_backtest(bt_occ)
display(met_occ.round(4))
assert bt_occ["y_pred"].dropna().between(0, 1).all()

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_scatter(x=occ.index, y=occ.values, name="Observado", line=dict(color="#0E7C86"))
for m, dash in zip(["logit_ets", "naive_estacional"], ["dash", "dot"]):
    d = bt_occ[(bt_occ.modelo == m) & bt_occ.valido]
    fig.add_scatter(x=d.fecha, y=d.y_pred, name=fc.ETIQUETAS[m], mode="lines+markers", line=dict(dash=dash, color="#7D3C98" if m == "logit_ets" else "#6C757D"), marker=dict(symbol="circle-open"))
fig.update_layout(title="Backtest rolling-origin · ocupación Izabal", yaxis_range=[0, 1.05])
fig.show()

### Pronóstico 2026 (Estimación)

In [ ]:
fut = pd.concat([fc.pronostico(occ, 12, m, bounds=(0, 1), migracion=migra) for m in fc.MODELOS]).dropna(subset=["estimacion"])
display(fut.pivot(index="fecha", columns="modelo", values="estimacion").round(3))
assert fut["estimacion"].between(0, 1).all()

## B. Encuesta: mes sin filas (2025-01) y backtest
⚠️ Con 6–117 registros/mes la serie es ruidosa: se reporta el error real. La transformación usada es `log1p` (conteos ≥ 0); el plan no la fija para la encuesta.

In [ ]:
bt_enc = fc.backtest(enc, transform="log", bounds=(0, np.inf), migracion=migra, desde="2024-01-01")
display(fc.metricas_backtest(bt_enc).round(3))
for m in fc.MODELOS:
    print(m, round(fc.imputar_mes(enc, "2025-01-01", m, transform="log", bounds=(0, np.inf), migracion=migra), 1))

## C. Pseudo-out-of-sample: ene–feb 2026 de la encuesta
Se enmascaran, se imputan con datos < t y se comparan contra el valor real.

In [ ]:
filas = []
for f in ["2026-01-01", "2026-02-01"]:
    real = enc.get(pd.Timestamp(f), np.nan)
    for m in fc.MODELOS:
        est = fc.imputar_mes(enc.drop(index=[i for i in enc.index if i >= pd.Timestamp(f)]), f, m, transform="log", bounds=(0, np.inf), migracion=migra)
        filas.append({"fecha": f, "modelo": m, "real": real, "estimacion": est, "error_%": 100 * abs(real - est) / real if real else np.nan})
display(pd.DataFrame(filas).round(2))

## Salidas procesadas
`python -m src.etl.pipeline` escribe `forecast_occupancy`, `forecast_survey` y `forecast_metrics` en `data/processed/`.

In [ ]:
from src.data import load
for n in ["forecast_occupancy", "forecast_survey", "forecast_metrics"]:
    print(n, load(n).shape)